# Prep data set to pass off for training

### Creating 3 datasets
1) Text = Metadata only, 2) Text = Transcript, 3) Text = Title, Description, Transcript 

In [10]:
import pandas as pd
import os

In [11]:
input = r'../../data/mccray/split_data/h2_split/D_mcc_h2_main.csv'
output_dir = "../../data/mccray/split_data/h2_split/training/"
df = pd.read_csv(input)
# list(df)

In [12]:
# Transcripts
og_tr = "Original Transcript"
sc_tr_v1 = 'D_mcc_cleaned_v1'
sc_tr_v2 = 'D_mcc_cleaned_v2'

# location = "clean"
# tr_type = sc_tr_v2  

location = "original"
tr_type = og_tr  

# Helper function to create formatted text field
def create_text_field(row, fields):
    """Create formatted text field from specified columns"""
    text_parts = [] 
    for field in fields:
        if field in row and pd.notna(row[field]) and str(row[field]).strip():
            # Format as [Field]: "content"
            content = str(row[field]).strip()
            text_parts.append(f"[{field}]: \"{content}\"")
    return "\n".join(text_parts)

# 1. Title, Description, and Date Fields only
title_description = df.copy()
title_description['text'] = title_description.apply(
    lambda row: create_text_field(row, ['Title', 'Description', 'Date']), axis=1
)
title_description = title_description[['Reference URL', 'Year', 'text']].copy()

# 2. Transcript field only
clean = df.copy()
clean['text'] = clean[tr_type].fillna('').astype(str)
# Keep only essential columns
clean = clean[['Reference URL', 'Year', 'text']].copy()

# 3. Title, Description, Date, and Transcript Fields (with Date before Transcript)
all_meta = df.copy()
# Create the combined text field with Date before Transcript
all_meta['text'] = all_meta.apply(
    lambda row: create_text_field(row, ['Title', 'Description', 'Date']) + 
                f"\n[Transcript]: \"{str(row[tr_type]).strip()}\"" 
                if pd.notna(row[tr_type]) and str(row[tr_type]).strip() 
                else create_text_field(row, ['Title', 'Description', 'Date']), 
    axis=1
)
# Keep only essential columns
all_meta = all_meta[['Reference URL', 'Year', 'text']].copy()

title_description.to_csv(f"{output_dir}/{location}/D_mcc_h2_title_description_date.csv", index=False)
clean.to_csv(f"{output_dir}/{location}/D_mcc_h2_transcript_only.csv", index=False)
all_meta.to_csv(f"{output_dir}/{location}/D_mcc_h2_all_metadata.csv", index=False)

print(f"\n=== DATASETS SAVED ===")
print(f"- Title + Description + Date: {output_dir}/{location}/D_mcc_h2_title_description_date.csv")
print(f"- Transcript only: {output_dir}/{location}/D_mcc_h2_transcript_only.csv")
print(f"- All metadata: {output_dir}/{location}/D_mcc_h2_all_metadata.csv")

# Additional step: Filter for 1940-1950s
print(f"\n=== FILTERING FOR 1940s ===")
def filter_1940s(df):
    # Convert Year to numeric, handle any non-numeric values
    df_copy = df.copy()
    df_copy['Year'] = pd.to_numeric(df_copy['Year'], errors='coerce')
    # Filter for 1940-1950
    return df_copy[(df_copy['Year'] >= 1940) & (df_copy['Year'] <= 1950)]

title_description_1940s = filter_1940s(title_description)
tr_type_1940s = filter_1940s(clean)
all_meta_1940s = filter_1940s(all_meta)

print(f"1940s filtering results:")
print(f"-  Title + Description + Date: {len(title_description)} -> {len(title_description_1940s)} rows")
print(f"- Transcript only: {len(clean)} -> {len(tr_type_1940s)} rows")
print(f"- All metadata: {len(all_meta)} -> {len(all_meta_1940s)} rows")

# Save 1940s datasets
title_description_1940s.to_csv(f"{output_dir}/{location}/1940s/D_mcc_h2_title_description_date_1940s.csv", index=False)
tr_type_1940s.to_csv(f"{output_dir}/{location}/1940s/D_mcc_h2_transcript_only_1940s.csv", index=False)
all_meta_1940s.to_csv(f"{output_dir}/{location}/1940s/D_mcc_h2_all_metadata_1940s.csv", index=False)

print(f"\n=== 1940s DATASETS SAVED ===")
print(f"- Title + Description + Date (1940s): {output_dir}/{location}/1940s/D_mcc_h2_title_description_date_1940s.csv")
print(f"- Transcript only (1940s): {output_dir}/{location}/1940s/D_mcc_h2_transcript_only_1940s.csv")
print(f"- All metadata (1940s): {output_dir}/{location}/1940s/D_mcc_h2_all_metadata_1940s.csv")



=== DATASETS SAVED ===
- Title + Description + Date: ../../data/mccray/split_data/h2_split/training//original/D_mcc_h2_title_description_date.csv
- Transcript only: ../../data/mccray/split_data/h2_split/training//original/D_mcc_h2_transcript_only.csv
- All metadata: ../../data/mccray/split_data/h2_split/training//original/D_mcc_h2_all_metadata.csv

=== FILTERING FOR 1940s ===
1940s filtering results:
-  Title + Description + Date: 12752 -> 3649 rows
- Transcript only: 12752 -> 3649 rows
- All metadata: 12752 -> 3649 rows

=== 1940s DATASETS SAVED ===
- Title + Description + Date (1940s): ../../data/mccray/split_data/h2_split/training//original/1940s/D_mcc_h2_title_description_date_1940s.csv
- Transcript only (1940s): ../../data/mccray/split_data/h2_split/training//original/1940s/D_mcc_h2_transcript_only_1940s.csv
- All metadata (1940s): ../../data/mccray/split_data/h2_split/training//original/1940s/D_mcc_h2_all_metadata_1940s.csv


In [13]:
# Display sample outputs
print("\n=== SAMPLE OUTPUTS ===")
print(f"\n1. Title + Description + Date dataset: {len(title_description)} rows")
print("Sample:")
print(title_description['text'].iloc[0][:200] + "...")

print(f"\n2. Transcript only dataset: {len(clean)} rows")
print("Sample:")
print(clean['text'].iloc[0][:200] + "...")

print(f"\n3. All metadata dataset: {len(all_meta)} rows")
print("Sample:")
print(all_meta['text'].iloc[0][:300] + "...")


=== SAMPLE OUTPUTS ===

1. Title + Description + Date dataset: 12752 rows
Sample:
[Title]: "Afro-American Newsboy Application signed by Mrs. C. B. Berry"
[Description]: "An application to be a Newsboy for the Afro-American submitted by Mrs. C. B. Berry."...

2. Transcript only dataset: 12752 rows
Sample:
-5491  AFRO-AMERICAN  NEWSBOY'S  APPLICATION  I hereby apply for membership in the AFRO Newsboys' Association of Carriers  and Street Salesmen, and if accepted, this application being properly signed ...

3. All metadata dataset: 12752 rows
Sample:
[Title]: "Afro-American Newsboy Application signed by Mrs. C. B. Berry"
[Description]: "An application to be a Newsboy for the Afro-American submitted by Mrs. C. B. Berry."
[Transcript]: "-5491  AFRO-AMERICAN  NEWSBOY'S  APPLICATION  I hereby apply for membership in the AFRO Newsboys' Association of...


In [14]:
# df.to_csv(f"../../data/mccray/split_data/h2_split/training/D_mcc_h2_training.csv", index=False)